In [1]:
# ─────────────────────────────────────────────
# 종합 실습용 데이터 — 새 스냅샷 (이 셀부터 단독 실행 가능)
# ─────────────────────────────────────────────
import numpy as np
import pandas as pd

np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})
print("스냅샷 준비 완료 — orders:", ordr.shape, "| customers:", cust.shape, "| products:", prod.shape)

스냅샷 준비 완료 — orders: (1500, 6) | customers: (200, 3) | products: (30, 3)


In [2]:
cust.head()

,customer_id,region,membership
0,C0001,인천,basic
1,C0002,서울,basic
2,C0003,경기,premium
3,C0004,부산,basic
4,C0005,인천,basic


In [3]:
prod.head()

,product_id,category,price
0,P001,식품,25000
1,P002,식품,40000
2,P003,패션,12000
3,P004,식품,12000
4,P005,패션,40000


In [4]:
ordr.head()

,order_id,customer_id,product_id,quantity,amount,order_date
0,O00001,C0032,P005,3,120000.0,2025-04-12
1,O00002,C0049,P020,1,25000.0,2025-03-09
2,O00003,C0041,P016,3,36000.0,2025-01-27
3,O00004,C0123,P002,3,120000.0,2025-03-26
4,O00005,C0150,P014,3,120000.0,2025-03-16


In [5]:
cust["customer_id"].duplicated().sum()

np.int64(0)

In [6]:
prod["product_id"].duplicated().sum()

np.int64(0)

In [7]:
ordr[ordr["customer_id"] == "C0090"]

,order_id,customer_id,product_id,quantity,amount,order_date
14,O00015,C0090,P018,3,120000.0,2025-01-24
16,O00017,C0090,P001,1,25000.0,2025-03-30
82,O00083,C0090,P018,1,40000.0,2025-03-22
105,O00106,C0090,P020,1,25000.0,2025-05-27
328,O00329,C0090,P021,2,24000.0,2025-04-20
658,O00659,C0090,P030,1,12000.0,2025-01-26
700,O00701,C0090,P017,2,150000.0,2025-02-12
721,O00722,C0090,P013,2,80000.0,2025-03-16
1001,O01002,C0090,P001,3,75000.0,2025-04-04
1185,O01186,C0090,P024,1,75000.0,2025-01-17


In [8]:
ordr["product_id"].duplicated().sum()

np.int64(1470)

In [9]:
ordr_cust = pd.merge(ordr, cust, on = "customer_id", how = "left", validate = "m:1")
ordr_cust.head()

,order_id,customer_id,product_id,quantity,amount,order_date,region,membership
0,O00001,C0032,P005,3,120000.0,2025-04-12,서울,premium
1,O00002,C0049,P020,1,25000.0,2025-03-09,서울,basic
2,O00003,C0041,P016,3,36000.0,2025-01-27,인천,basic
3,O00004,C0123,P002,3,120000.0,2025-03-26,경기,basic
4,O00005,C0150,P014,3,120000.0,2025-03-16,경기,basic


In [10]:
all_data = pd.merge(ordr_cust, prod, on = "product_id", how = "left", validate = "m:1")
all_data.head()

,order_id,customer_id,product_id,quantity,amount,order_date,region,membership,category,price
0,O00001,C0032,P005,3,120000.0,2025-04-12,서울,premium,패션,40000
1,O00002,C0049,P020,1,25000.0,2025-03-09,서울,basic,식품,25000
2,O00003,C0041,P016,3,36000.0,2025-01-27,인천,basic,패션,12000
3,O00004,C0123,P002,3,120000.0,2025-03-26,경기,basic,식품,40000
4,O00005,C0150,P014,3,120000.0,2025-03-16,경기,basic,가전,40000


In [11]:
all_data[all_data.isna().any(axis=1)]

,order_id,customer_id,product_id,quantity,amount,order_date,region,membership,category,price


In [31]:
print("월별 KPI")
all_data.groupby("month").agg(매출합계 = ("amount", "sum"),
                              평균금액 = ("amount", "mean"),
                              주문건수 = ("order_id", "count"))

월별 KPI


,매출합계,평균금액,주문건수
month,,,
1,19902000.0,60309.090909,330
2,17043000.0,58972.318339,289
3,20035000.0,61457.055215,326
4,17602000.0,61331.010453,287
5,16374000.0,61097.014925,268


In [32]:
print("지역별 KPI")
all_data.groupby("region").agg(매출합계 = ("amount", "sum"),
                              평균금액 = ("amount", "mean"),
                              주문건수 = ("order_id", "count"))

지역별 KPI


,매출합계,평균금액,주문건수
region,,,
경기,21145000.0,66078.125000,320
부산,18754000.0,58789.968652,319
서울,23480000.0,59143.576826,397
인천,27577000.0,59433.189655,464


In [33]:
print("카테고리별 KPI")
all_data.groupby("category").agg(매출합계 = ("amount", "sum"),
                              평균금액 = ("amount", "mean"),
                              주문건수 = ("order_id", "count"))

카테고리별 KPI


,매출합계,평균금액,주문건수
category,,,
가전,20248000.0,50874.371859,398
뷰티,9986000.0,52010.416667,192
식품,22404000.0,56291.457286,398
패션,38318000.0,74839.843750,512


In [42]:
monthly_sum = all_data.groupby("month")["amount"].sum()
monthly_share = (monthly_sum / monthly_sum.sum() * 100).round(1)
print(f"점유율 분석 \n {monthly_share}")

점유율 분석 
 month
1    21.9
2    18.7
3    22.0
4    19.4
5    18.0
Name: amount, dtype: float64


In [52]:
print("객단가 확인")
print(all_data["amount"].mean().round(0))

객단가 확인
60637.0


In [54]:
print("멤버쉽별 KPI")
all_data.groupby("membership").agg(매출합계 = ("amount", "sum"),
                              평균금액 = ("amount", "mean"),
                              주문건수 = ("order_id", "count"))

멤버쉽별 KPI


,매출합계,평균금액,주문건수
membership,,,
basic,55411000.0,59517.722879,931
premium,27002000.0,61090.497738,442
vip,8543000.0,67267.716535,127


In [55]:
membership_sum = all_data.groupby("membership")["amount"].sum()
membership_share = (membership_sum / membership_sum.sum() * 100).round(1)
print(f"점유율 분석 \n {membership_share}")

점유율 분석 
 membership
basic      60.9
premium    29.7
vip         9.4
Name: amount, dtype: float64


In [63]:
all_data["membership"].value_counts()

membership
basic      931
premium    442
vip        127
Name: count, dtype: int64